<a href="https://colab.research.google.com/github/alperdinceler/Machine-learning/blob/main/kaggle_yar%C4%B1%C5%9Fma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import OrdinalEncoder

# 1. Verileri yükleme
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

# 2. Veri Ön İşleme (Data Preprocessing)
# 'totalcharges' sütunundaki boşlukları (space) NaN ile değiştirip numerik formata çevirme
train['totalcharges'] = pd.to_numeric(train['totalcharges'].replace(r'^\s*$', np.nan, regex=True))
test['totalcharges'] = pd.to_numeric(test['totalcharges'].replace(r'^\s*$', np.nan, regex=True))

# Eksik (NaN) değerleri 0 ile doldurma
train['totalcharges'] = train['totalcharges'].fillna(0)
test['totalcharges'] = test['totalcharges'].fillna(0)

# Kategorik sütunları (metin verilerini) belirleme
cat_cols = train.select_dtypes(include=['object']).columns.tolist()
cat_cols.remove('customerid') # Müşteri ID'si model için bir özellik (feature) değildir

# Kategorik verileri sayısal verilere dönüştürme (Ordinal Encoding)
oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
train[cat_cols] = oe.fit_transform(train[cat_cols].astype(str))
test[cat_cols] = oe.transform(test[cat_cols].astype(str))

# 3. Veriyi Modele Hazırlama
X_train = train.drop(['customerid', 'churn'], axis=1) # Eğitim özellikleri
y_train = train['churn']                              # Eğitim hedef değişkeni
X_test = test.drop(['customerid'], axis=1)            # Test özellikleri

# 4. Modeli Eğitme
# Dengesiz ve tablosal verilerde başarılı olan HistGradientBoosting algoritması kullanıldı
model = HistGradientBoostingClassifier(random_state=42, max_iter=100, learning_rate=0.05)
model.fit(X_train, y_train)

# 5. Tahmin (Prediction)
# predict_proba() fonksiyonu ile [Hizmette Kalma Olasılığı, Ayrılma Olasılığı] hesaplanır.
# Bize 1 sınıfı, yani ayrılma olasılığı ([:, 1]) lazım.
preds = model.predict_proba(X_test)[:, 1]

# 6. Sonuçları Kaydetme
sub = pd.DataFrame({
    'customerid': test['customerid'],
    'churn': preds
})

# Dosyayı dışarı aktarma
sub.to_csv('submission.csv', index=False)
print("Tahminler 'submission.csv' dosyasına başarıyla kaydedildi!")

Tahminler 'submission.csv' dosyasına başarıyla kaydedildi!


In [ ]:
df.head()

,customerid,gender,seniorcitizen,partner,dependents,tenure,phoneservice,multiplelines,internetservice,onlinesecurity,onlinebackup,deviceprotection,techsupport,streamingtv,streamingmovies,contract,paperlessbilling,paymentmethod,monthlycharges,totalcharges
0,4376-KFVRS,Male,0,Yes,Yes,72,Yes,Yes,Fiber optic,Yes,Yes,Yes,Yes,Yes,Yes,Two year,Yes,Credit card (automatic),114.05,8468.20
1,2754-SDJRD,Female,1,No,No,8,Yes,Yes,Fiber optic,No,No,No,Yes,Yes,Yes,Month-to-month,Yes,Credit card (automatic),100.15,908.55
2,9917-KWRBE,Female,0,Yes,Yes,41,Yes,Yes,DSL,Yes,Yes,Yes,No,Yes,No,One year,Yes,Credit card (automatic),78.35,3211.20
3,0365-GXEZS,Male,0,Yes,No,18,Yes,No,Fiber optic,No,No,Yes,Yes,No,No,Month-to-month,No,Electronic check,78.20,1468.75
4,9385-NXKDA,Female,0,Yes,No,72,Yes,Yes,DSL,Yes,Yes,Yes,No,Yes,Yes,Two year,Yes,Credit card (automatic),82.65,5919.35


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import OrdinalEncoder

# 1. Verileri yükleme
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

# --- VERİ TEMİZLİĞİ ---
train['totalcharges'] = pd.to_numeric(train['totalcharges'].replace(r'^\s*$', np.nan, regex=True)).fillna(0)
test['totalcharges'] = pd.to_numeric(test['totalcharges'].replace(r'^\s*$', np.nan, regex=True)).fillna(0)

# Verileri kolay işlemek için birleştiriyoruz (işlem bitince ayıracağız)
# Not: churn sütununu ayırıp sadece özellikleri birleştiriyoruz
y_train = train['churn']
all_data = pd.concat([train.drop(['customerid', 'churn'], axis=1), test.drop(['customerid'], axis=1)], axis=0)

# --- 🌟 STRATEJİ 1: ÖZELLİK MÜHENDİSLİĞİ (FEATURE ENGINEERING) 🌟 ---

# A. Aylık Ücrete Göre Yaklaşık Kalma Süresi (Tenure Estimation)
# (Tenure sütunu zaten var olabilir ama bazen bu oran ekstra bilgi sağlar)
all_data['tenure_calc'] = all_data['totalcharges'] / (all_data['monthlycharges'] + 1e-5) # Sıfıra bölmeyi engellemek için küçük bir sayı ekliyoruz

# B. Toplam Ek Servis Sayısı (Müşteri ne kadar çok servise bağlı?)
services = ['onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport', 'streamingtv', 'streamingmovies']
# Bu servislerde 'Yes' diyenlerin sayısını topluyoruz
all_data['total_services'] = all_data[services].apply(lambda x: (x == 'Yes').sum(), axis=1)

# C. Aile Büyüklüğü (Partner ve Bakmakla yükümlü olunan kişiler)
all_data['is_alone'] = ((all_data['partner'] == 'No') & (all_data['dependents'] == 'No')).astype(int)

# --- KATEGORİK VERİLERİ DÖNÜŞTÜRME ---
cat_cols = all_data.select_dtypes(include=['object']).columns.tolist()
oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
all_data[cat_cols] = oe.fit_transform(all_data[cat_cols].astype(str))

# --- VERİYİ TEKRAR TRAIN VE TEST OLARAK AYIRMA ---
X_train = all_data.iloc[:len(train)]
X_test = all_data.iloc[len(train):]

# --- MODELLİ EĞİTME ---
# learning_rate'i biraz düşürüp max_iter (ağaç sayısını) artırmak genelde daha iyi sonuç verir
model = HistGradientBoostingClassifier(
    random_state=42,
    max_iter=250,        # 100'den 250'ye çıkarıldı
    learning_rate=0.03,  # 0.05'ten 0.03'e düşürüldü (daha yavaş ama daha sağlam öğrenir)
    l2_regularization=1.5 # Aşırı öğrenmeyi (overfitting) engellemek için regülarizasyon eklendi
)

model.fit(X_train, y_train)

# --- TAHMİN VE KAYIT ---
preds = model.predict_proba(X_test)[:, 1]

sub = pd.DataFrame({
    'customerid': test['customerid'],
    'churn': preds
})
sub.to_csv('submission_v2_feature_engineering.csv', index=False)
print("Yeni tahminler 'submission_v2_feature_engineering.csv' dosyasına kaydedildi!")

Yeni tahminler 'submission_v2_feature_engineering.csv' dosyasına kaydedildi!


In [ ]:
!pip install xgboost lightgbm catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 6.5 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# 1. Verileri Yükleme
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

# --- VERİ TEMİZLİĞİ VE ÖZELLİK MÜHENDİSLİĞİ ---
train['totalcharges'] = pd.to_numeric(train['totalcharges'].replace(r'^\s*$', np.nan, regex=True)).fillna(0)
test['totalcharges'] = pd.to_numeric(test['totalcharges'].replace(r'^\s*$', np.nan, regex=True)).fillna(0)

y_train = train['churn']
all_data = pd.concat([train.drop(['customerid', 'churn'], axis=1), test.drop(['customerid'], axis=1)], axis=0)

# Yeni Özellikler
all_data['tenure_calc'] = all_data['totalcharges'] / (all_data['monthlycharges'] + 1e-5)
services = ['onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport', 'streamingtv', 'streamingmovies']
all_data['total_services'] = all_data[services].apply(lambda x: (x == 'Yes').sum(), axis=1)
all_data['is_alone'] = ((all_data['partner'] == 'No') & (all_data['dependents'] == 'No')).astype(int)

# Kategorik Verileri Kodlama
cat_cols = all_data.select_dtypes(include=['object']).columns.tolist()
oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
all_data[cat_cols] = oe.fit_transform(all_data[cat_cols].astype(str))

# Train ve Test'i Geri Ayırma
X_train = all_data.iloc[:len(train)].reset_index(drop=True)
X_test = all_data.iloc[len(train):].reset_index(drop=True)

# --- 🌟 STRATEJİ: K-FOLD CROSS VALIDATION & ENSEMBLE 🌟 ---
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# Test tahminlerini tutacağımız boş dizi
test_predictions = np.zeros(len(X_test))

print("Modeller eğitiliyor, lütfen bekleyin...\n")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    print(f"--- Fold {fold + 1} ---")

    # Eğitim ve Doğrulama setlerini ayırma
    X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
    X_val, y_val = X_train.iloc[val_idx], y_train.iloc[val_idx]

    # 1. LightGBM Modeli
    model_lgb = LGBMClassifier(n_estimators=300, learning_rate=0.03, random_state=42, verbose=-1)
    model_lgb.fit(X_tr, y_tr)
    preds_lgb = model_lgb.predict_proba(X_test)[:, 1]

    # 2. XGBoost Modeli
    model_xgb = XGBClassifier(n_estimators=300, learning_rate=0.03, random_state=42, use_label_encoder=False, eval_metric='logloss')
    model_xgb.fit(X_tr, y_tr)
    preds_xgb = model_xgb.predict_proba(X_test)[:, 1]

    # 3. CatBoost Modeli (Sessiz modda çalışır)
    model_cat = CatBoostClassifier(iterations=300, learning_rate=0.03, random_state=42, verbose=0)
    model_cat.fit(X_tr, y_tr)
    preds_cat = model_cat.predict_proba(X_test)[:, 1]

    # Modellerin Tahminlerini Harmanlama (Blending - Eşit Ağırlık)
    fold_preds = (preds_lgb + preds_xgb + preds_cat) / 3.0

    # Bu fold'dan gelen test tahminlerini ana toplama ekleme
    test_predictions += fold_preds / N_SPLITS

# --- SONUÇLARI KAYDETME ---
sub = pd.DataFrame({
    'customerid': test['customerid'],
    'churn': test_predictions
})
sub.to_csv('submission_v3_ensemble.csv', index=False)
print("\nHARİKA! 'submission_v3_ensemble.csv' dosyası yarışmaya gönderilmeye hazır.")

Modeller eğitiliyor, lütfen bekleyin...

--- Fold 1 ---


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [10:59:36] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


--- Fold 2 ---


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [10:59:45] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


--- Fold 3 ---


...

--- Fold 4 ---


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [11:00:01] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


--- Fold 5 ---


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [11:00:04] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



HARİKA! 'submission_v3_ensemble.csv' dosyası yarışmaya gönderilmeye hazır.


In [ ]:
!pip install optuna xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 14.6 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import optuna
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

# 1. Verileri Yükleme ve Hazırlama (Öncekiyle aynı temizlik)
train = pd.read_csv('train.csv')
train['totalcharges'] = pd.to_numeric(train['totalcharges'].replace(r'^\s*$', np.nan, regex=True)).fillna(0)

# Özellik Mühendisliği
train['tenure_calc'] = train['totalcharges'] / (train['monthlycharges'] + 1e-5)
services = ['onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport', 'streamingtv', 'streamingmovies']
train['total_services'] = train[services].apply(lambda x: (x == 'Yes').sum(), axis=1)
train['is_alone'] = ((train['partner'] == 'No') & (train['dependents'] == 'No')).astype(int)

y = train['churn']
X = train.drop(['customerid', 'churn'], axis=1)

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X[cat_cols] = oe.fit_transform(X[cat_cols].astype(str))

# 2. Optuna Amaç Fonksiyonu (Objective Function)
def objective(trial):
    # Optuna'nın deneyeceği parametre aralıkları (Özellikle Overfitting'i engelleyenler)
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),  # L1 Regülarizasyon (Ezber bozucu)
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True), # L2 Regülarizasyon (Ezber bozucu)
        'random_state': 42,
        'eval_metric': 'auc',
        'use_label_encoder': False
    }

    # 5-Fold Cross Validation ile test etme
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = []

    for train_idx, val_idx in skf.split(X, y):
        X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

        model = XGBClassifier(**param)
        model.fit(X_tr, y_tr, verbose=False)

        preds = model.predict_proba(X_val)[:, 1]
        cv_scores.append(roc_auc_score(y_val, preds))

    return np.mean(cv_scores)

# 3. Optimizasyonu Başlatma
print("Optuna hiperparametre avı başlıyor! (Bu işlem biraz sürebilir...)")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30) # 30 farklı kombinasyon deneyecek

print("\n--- OPTİMAİZASYON TAMAMLANDI ---")
print(f"En İyi ROC AUC Skoru: {study.best_value:.5f}")
print("En İyi Parametreler:")
for key, value in study.best_params.items():
    print(f"    '{key}': {value},")

[I 2026-05-20 11:09:45,561] A new study created in memory with name: no-name-390b0c56-f78e-409b-8243-ec933e332522


Optuna hiperparametre avı başlıyor! (Bu işlem biraz sürebilir...)


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [11:09:45] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [11:09:49] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [11:09:53] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [11:09:59] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:


--- OPTİMAİZASYON TAMAMLANDI ---
En İyi ROC AUC Skoru: 0.84984
En İyi Parametreler:
    'n_estimators': 298,
    'learning_rate': 0.015271002266249084,
    'max_depth': 3,
    'min_child_weight': 7,
    'subsample': 0.8684346870701609,
    'colsample_bytree': 0.6290071086268502,
    'reg_alpha': 0.0008202745022175132,
    'reg_lambda': 0.010159518749251518,


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# 1. Verileri Yükleme ve Hazırlama
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

train['totalcharges'] = pd.to_numeric(train['totalcharges'].replace(r'^\s*$', np.nan, regex=True)).fillna(0)
test['totalcharges'] = pd.to_numeric(test['totalcharges'].replace(r'^\s*$', np.nan, regex=True)).fillna(0)

y_train = train['churn']
all_data = pd.concat([train.drop(['customerid', 'churn'], axis=1), test.drop(['customerid'], axis=1)], axis=0)

# Özellik Mühendisliği
all_data['tenure_calc'] = all_data['totalcharges'] / (all_data['monthlycharges'] + 1e-5)
services = ['onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport', 'streamingtv', 'streamingmovies']
all_data['total_services'] = all_data[services].apply(lambda x: (x == 'Yes').sum(), axis=1)
all_data['is_alone'] = ((all_data['partner'] == 'No') & (all_data['dependents'] == 'No')).astype(int)

# Kategorik Verileri Kodlama
cat_cols = all_data.select_dtypes(include=['object']).columns.tolist()
oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
all_data[cat_cols] = oe.fit_transform(all_data[cat_cols].astype(str))

X_train = all_data.iloc[:len(train)].reset_index(drop=True)
X_test = all_data.iloc[len(train):].reset_index(drop=True)

# --- 🌟 OPTUNA İLE OPTİMİZE EDİLMİŞ ŞAMPİYON ENSEMBLE MODELİ 🌟 ---
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
test_predictions = np.zeros(len(X_test))

print("Optuna ayarlarıyla modeller eğitiliyor...\n")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]

    # 1. OPTUNA'NIN BULDUĞU KUSURSUZ XGBOOST MODELİ (Ağırlığı: %40)
    model_xgb = XGBClassifier(
        n_estimators=298,
        learning_rate=0.015271002266249084,
        max_depth=3,
        min_child_weight=7,
        subsample=0.8684346870701609,
        colsample_bytree=0.6290071086268502,
        reg_alpha=0.0008202745022175132,
        reg_lambda=0.010159518749251518,
        random_state=42,
        eval_metric='auc'
    )
    model_xgb.fit(X_tr, y_tr)
    preds_xgb = model_xgb.predict_proba(X_test)[:, 1]

    # 2. XGBOOST'A UYUMLANMIŞ KATI LIGHTGBM MODELİ (Ağırlığı: %30)
    model_lgb = LGBMClassifier(
        n_estimators=300, learning_rate=0.015, max_depth=3, num_leaves=7,
        reg_lambda=5.0, subsample=0.8, colsample_bytree=0.6, random_state=42, verbose=-1
    )
    model_lgb.fit(X_tr, y_tr)
    preds_lgb = model_lgb.predict_proba(X_test)[:, 1]

    # 3. XGBOOST'A UYUMLANMIŞ KATI CATBOOST MODELİ (Ağırlığı: %30)
    model_cat = CatBoostClassifier(
        iterations=300, learning_rate=0.015, depth=3, l2_leaf_reg=10.0,
        random_state=42, verbose=0
    )
    model_cat.fit(X_tr, y_tr)
    preds_cat = model_cat.predict_proba(X_test)[:, 1]

    # Ağırlıklı Harmanlama (XGBoost'a güvenimiz tam olduğu için %40 güç verdik)
    fold_preds = (preds_xgb * 0.40) + (preds_lgb * 0.30) + (preds_cat * 0.30)
    test_predictions += fold_preds / N_SPLITS

# --- SONUÇLARI KAYDETME ---
sub = pd.DataFrame({
    'customerid': test['customerid'],
    'churn': test_predictions
})
sub.to_csv('submission_v5_optuna.csv', index=False)
print("HARİKA! 'submission_v5_optuna.csv' dosyası Kaggle'a gönderilmeye hazır.")

Optuna ayarlarıyla modeller eğitiliyor...

HARİKA! 'submission_v5_optuna.csv' dosyası Kaggle'a gönderilmeye hazır.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# 1. Verileri Yükleme ve Hazırlama
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

train['totalcharges'] = pd.to_numeric(train['totalcharges'].replace(r'^\s*$', np.nan, regex=True)).fillna(0)
test['totalcharges'] = pd.to_numeric(test['totalcharges'].replace(r'^\s*$', np.nan, regex=True)).fillna(0)

y_train = train['churn']
all_data = pd.concat([train.drop(['customerid', 'churn'], axis=1), test.drop(['customerid'], axis=1)], axis=0)

# --- 🌟 YENİ ALTIN ÖZELLİKLER (GOLDEN FEATURES) 🌟 ---
all_data['tenure_calc'] = all_data['totalcharges'] / (all_data['monthlycharges'] + 1e-5)

# Gizli Ücret Farkı (Gerçek Ödenen - Olması Gereken)
all_data['extra_charges'] = all_data['totalcharges'] - (all_data['tenure'] * all_data['monthlycharges'])

# Otomatik Ödeme Yapanların Sadakati (Otomatik ödeyenler genelde sistemde kalır)
all_data['is_automatic_payment'] = all_data['paymentmethod'].str.contains('automatic', case=False, na=False).astype(int)

# Servis Kullanım Yoğunluğu
services = ['onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport', 'streamingtv', 'streamingmovies']
all_data['total_services'] = all_data[services].apply(lambda x: (x == 'Yes').sum(), axis=1)

# Aile/Yalnızlık Durumu
all_data['is_alone'] = ((all_data['partner'] == 'No') & (all_data['dependents'] == 'No')).astype(int)

# Kategorik Verileri Kodlama
cat_cols = all_data.select_dtypes(include=['object']).columns.tolist()
oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
all_data[cat_cols] = oe.fit_transform(all_data[cat_cols].astype(str))

X_train = all_data.iloc[:len(train)].reset_index(drop=True)
X_test = all_data.iloc[len(train):].reset_index(drop=True)

# --- 🌟 SEED BLENDING (3 FARKLI TOHUMLA HARMANLAMA) 🌟 ---
SEEDS = [42, 2024, 777] # Modelleri bu 3 farklı rastgelelik senaryosunda eğiteceğiz
N_SPLITS = 5
final_test_predictions = np.zeros(len(X_test))

print("Seed Blending (3 Farklı Senaryo) başlıyor. Lütfen bekleyin...\n")

for seed in SEEDS:
    print(f"--- Modeller Senaryo {seed} için eğitiliyor ---")
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    seed_predictions = np.zeros(len(X_test))

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
        X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]

        # XGBoost (Optuna Ayarları + Seed)
        model_xgb = XGBClassifier(
            n_estimators=298, learning_rate=0.01527, max_depth=3, min_child_weight=7,
            subsample=0.868, colsample_bytree=0.629, reg_alpha=0.0008, reg_lambda=0.0101,
            random_state=seed, eval_metric='auc'
        )
        model_xgb.fit(X_tr, y_tr)
        preds_xgb = model_xgb.predict_proba(X_test)[:, 1]

        # LightGBM (Güçlü Regülarizasyon + Seed)
        model_lgb = LGBMClassifier(
            n_estimators=300, learning_rate=0.015, max_depth=3, num_leaves=7,
            reg_lambda=5.0, subsample=0.8, colsample_bytree=0.6, random_state=seed, verbose=-1
        )
        model_lgb.fit(X_tr, y_tr)
        preds_lgb = model_lgb.predict_proba(X_test)[:, 1]

        # CatBoost (Güçlü Regülarizasyon + Seed)
        model_cat = CatBoostClassifier(
            iterations=300, learning_rate=0.015, depth=3, l2_leaf_reg=10.0,
            random_state=seed, verbose=0
        )
        model_cat.fit(X_tr, y_tr)
        preds_cat = model_cat.predict_proba(X_test)[:, 1]

        # Klasik Ağırlıklandırma
        fold_preds = (preds_xgb * 0.40) + (preds_lgb * 0.30) + (preds_cat * 0.30)
        seed_predictions += fold_preds / N_SPLITS

    final_test_predictions += seed_predictions / len(SEEDS)

# --- SONUÇLARI KAYDETME ---
sub = pd.DataFrame({
    'customerid': test['customerid'],
    'churn': final_test_predictions
})
sub.to_csv('submission_v6_final.csv', index=False)
print("\nHARİKA! 'submission_v6_final.csv' dosyası Kaggle'a gönderilmeye hazır.")

Seed Blending (3 Farklı Senaryo) başlıyor. Lütfen bekleyin...

--- Modeller Senaryo 42 için eğitiliyor ---
--- Modeller Senaryo 2024 için eğitiliyor ---
--- Modeller Senaryo 777 için eğitiliyor ---

HARİKA! 'submission_v6_final.csv' dosyası Kaggle'a gönderilmeye hazır.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# 1. Verileri Yükleme
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

train['totalcharges'] = pd.to_numeric(train['totalcharges'].replace(r'^\s*$', np.nan, regex=True)).fillna(0)
test['totalcharges'] = pd.to_numeric(test['totalcharges'].replace(r'^\s*$', np.nan, regex=True)).fillna(0)

y_train = train['churn']
all_data = pd.concat([train.drop(['customerid', 'churn'], axis=1), test.drop(['customerid'], axis=1)], axis=0)

# --- 🌟 SADECE 0.837 GETİREN TEMİZ ÖZELLİKLER (V5) 🌟 ---
all_data['tenure_calc'] = all_data['totalcharges'] / (all_data['monthlycharges'] + 1e-5)
services = ['onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport', 'streamingtv', 'streamingmovies']
all_data['total_services'] = all_data[services].apply(lambda x: (x == 'Yes').sum(), axis=1)
all_data['is_alone'] = ((all_data['partner'] == 'No') & (all_data['dependents'] == 'No')).astype(int)

# Kategorik Verileri Kodlama
cat_cols = all_data.select_dtypes(include=['object']).columns.tolist()
oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
all_data[cat_cols] = oe.fit_transform(all_data[cat_cols].astype(str))

X_train = all_data.iloc[:len(train)].reset_index(drop=True)
X_test = all_data.iloc[len(train):].reset_index(drop=True)

# --- 🌟 10-FOLD & 5-SEED BLENDING (MAXIMUM GÜVENLİK) 🌟 ---
N_SPLITS = 10 # 5'ten 10'a çıkarıldı (Modele daha çok eğitim verisi!)
SEEDS = [42, 2024, 777, 1990, 888] # 5 farklı şans faktörü
final_test_predictions = np.zeros(len(X_test))

print("150 Model eğitiliyor... Liderlik tablosu için son hazırlıklar...\n")

for seed in SEEDS:
    print(f"--- Senaryo Tohumu (Seed): {seed} Eğitiliyor ---")
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    seed_preds = np.zeros(len(X_test))

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
        X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]

        # 1. XGBoost (Optuna'nın Kusursuz Parametreleri)
        model_xgb = XGBClassifier(
            n_estimators=298, learning_rate=0.01527, max_depth=3, min_child_weight=7,
            subsample=0.868, colsample_bytree=0.629, reg_alpha=0.0008, reg_lambda=0.0101,
            random_state=seed, eval_metric='auc'
        )
        model_xgb.fit(X_tr, y_tr)
        preds_xgb = model_xgb.predict_proba(X_test)[:, 1]

        # 2. LightGBM (Güçlü Budanmış)
        model_lgb = LGBMClassifier(
            n_estimators=300, learning_rate=0.015, max_depth=3, num_leaves=7,
            reg_lambda=5.0, subsample=0.8, colsample_bytree=0.6, random_state=seed, verbose=-1
        )
        model_lgb.fit(X_tr, y_tr)
        preds_lgb = model_lgb.predict_proba(X_test)[:, 1]

        # 3. CatBoost (Aşırı Öğrenme Karşıtı Şampiyon)
        model_cat = CatBoostClassifier(
            iterations=300, learning_rate=0.015, depth=3, l2_leaf_reg=10.0,
            random_state=seed, verbose=0
        )
        model_cat.fit(X_tr, y_tr)
        preds_cat = model_cat.predict_proba(X_test)[:, 1]

        # Ağırlıklı Harmanlama (CatBoost ağırlığı artırıldı, en stabil model olduğu için)
        fold_preds = (preds_xgb * 0.40) + (preds_lgb * 0.25) + (preds_cat * 0.35)
        seed_preds += fold_preds / N_SPLITS

    final_test_predictions += seed_preds / len(SEEDS)

# --- SONUÇLARI KAYDETME ---
sub = pd.DataFrame({
    'customerid': test['customerid'],
    'churn': final_test_predictions
})
sub.to_csv('submission_v7.csv', index=False)
print("\n🔥 İŞLEM TAMAM! 'submission_v7_last_dance.csv' dosyası son hak için hazır!")

150 Model eğitiliyor... Liderlik tablosu için son hazırlıklar...

--- Senaryo Tohumu (Seed): 42 Eğitiliyor ---
--- Senaryo Tohumu (Seed): 2024 Eğitiliyor ---
--- Senaryo Tohumu (Seed): 777 Eğitiliyor ---
--- Senaryo Tohumu (Seed): 1990 Eğitiliyor ---
--- Senaryo Tohumu (Seed): 888 Eğitiliyor ---

🔥 İŞLEM TAMAM! 'submission_v7_last_dance.csv' dosyası son hak için hazır!


In [ ]:
import pandas as pd
import numpy as np
import optuna
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import warnings
warnings.filterwarnings("ignore")

# 1. VERİ YÜKLEME VE EN TEMİZ (0.837 GETİREN) ÖZELLİKLER
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

train['totalcharges'] = pd.to_numeric(train['totalcharges'].replace(r'^\s*$', np.nan, regex=True)).fillna(0)
test['totalcharges'] = pd.to_numeric(test['totalcharges'].replace(r'^\s*$', np.nan, regex=True)).fillna(0)

y = train['churn']
all_data = pd.concat([train.drop(['customerid', 'churn'], axis=1), test.drop(['customerid'], axis=1)], axis=0)

all_data['tenure_calc'] = all_data['totalcharges'] / (all_data['monthlycharges'] + 1e-5)
services = ['onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport', 'streamingtv', 'streamingmovies']
all_data['total_services'] = all_data[services].apply(lambda x: (x == 'Yes').sum(), axis=1)
all_data['is_alone'] = ((all_data['partner'] == 'No') & (all_data['dependents'] == 'No')).astype(int)

cat_cols = all_data.select_dtypes(include=['object']).columns.tolist()
oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
all_data[cat_cols] = oe.fit_transform(all_data[cat_cols].astype(str))

X = all_data.iloc[:len(train)].reset_index(drop=True)
X_test = all_data.iloc[len(train):].reset_index(drop=True)

# --- 1. LIGHTGBM OPTİMİZASYONU ---
def objective_lgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 400),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.05, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 5),
        'num_leaves': trial.suggest_int('num_leaves', 7, 31),
        'reg_lambda': trial.suggest_float('reg_lambda', 1.0, 10.0),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'random_state': 42,
        'verbose': -1
    }
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    for tr_idx, val_idx in skf.split(X, y):
        model = LGBMClassifier(**params)
        model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        preds = model.predict_proba(X.iloc[val_idx])[:, 1]
        scores.append(roc_auc_score(y.iloc[val_idx], preds))
    return np.mean(scores)

print("🎯 Adım 1/4: LightGBM optimize ediliyor...")
study_lgb = optuna.create_study(direction='maximize')
study_lgb.optimize(objective_lgb, n_trials=15) # Hızlı optimizasyon için 15 deneme
best_lgb_params = study_lgb.best_params
best_lgb_params['random_state'] = 42
best_lgb_params['verbose'] = -1

# --- 2. CATBOOST OPTİMİZASYONU ---
def objective_cat(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 100, 400),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.05, log=True),
        'depth': trial.suggest_int('depth', 3, 5),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 3.0, 15.0),
        'random_state': 42,
        'verbose': 0
    }
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    for tr_idx, val_idx in skf.split(X, y):
        model = CatBoostClassifier(**params)
        model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        preds = model.predict_proba(X.iloc[val_idx])[:, 1]
        scores.append(roc_auc_score(y.iloc[val_idx], preds))
    return np.mean(scores)

print("\n🎯 Adım 2/4: CatBoost optimize ediliyor...")
study_cat = optuna.create_study(direction='maximize')
study_cat.optimize(objective_cat, n_trials=15)
best_cat_params = study_cat.best_params
best_cat_params['random_state'] = 42
best_cat_params['verbose'] = 0

# --- 3. MODELLERİN OOF (Gerçek Test) TAHMİNLERİNİ ÇIKARMA ---
print("\n🎯 Adım 3/4: Bulunan kusursuz ayarlarla modeller eğitiliyor...")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_xgb = np.zeros(len(X)); test_xgb = np.zeros(len(X_test))
oof_lgb = np.zeros(len(X)); test_lgb = np.zeros(len(X_test))
oof_cat = np.zeros(len(X)); test_cat = np.zeros(len(X_test))

# Daha önce bulduğumuz harika XGBoost parametreleri
xgb_params = {'n_estimators': 298, 'learning_rate': 0.01527, 'max_depth': 3, 'min_child_weight': 7, 'subsample': 0.868, 'colsample_bytree': 0.629, 'reg_alpha': 0.0008, 'reg_lambda': 0.0101, 'random_state': 42, 'eval_metric': 'auc'}

for tr_idx, val_idx in skf.split(X, y):
    X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    m_xgb = XGBClassifier(**xgb_params); m_xgb.fit(X_tr, y_tr)
    oof_xgb[val_idx] = m_xgb.predict_proba(X_val)[:, 1]
    test_xgb += m_xgb.predict_proba(X_test)[:, 1] / 5

    m_lgb = LGBMClassifier(**best_lgb_params); m_lgb.fit(X_tr, y_tr)
    oof_lgb[val_idx] = m_lgb.predict_proba(X_val)[:, 1]
    test_lgb += m_lgb.predict_proba(X_test)[:, 1] / 5

    m_cat = CatBoostClassifier(**best_cat_params); m_cat.fit(X_tr, y_tr)
    oof_cat[val_idx] = m_cat.predict_proba(X_val)[:, 1]
    test_cat += m_cat.predict_proba(X_test)[:, 1] / 5

# --- 4. OPTUNA İLE AĞIRLIK (BLENDING) OPTİMİZASYONU ---
def objective_blend(trial):
    w_xgb = trial.suggest_float('w_xgb', 0.0, 1.0)
    w_lgb = trial.suggest_float('w_lgb', 0.0, 1.0)
    w_cat = trial.suggest_float('w_cat', 0.0, 1.0)

    # Ağırlıkların toplamının 1 (Yani %100) olmasını garantiliyoruz
    total = w_xgb + w_lgb + w_cat
    w_xgb /= total; w_lgb /= total; w_cat /= total

    blend_oof = (oof_xgb * w_xgb) + (oof_lgb * w_lgb) + (oof_cat * w_cat)
    return roc_auc_score(y, blend_oof)

print("\n🎯 Adım 4/4: Optuna en yüksek skoru verecek birleştirme oranlarını (Ağırlıkları) hesaplıyor...")
study_blend = optuna.create_study(direction='maximize')
study_blend.optimize(objective_blend, n_trials=100) # Tahminler elimizde olduğu için çok hızlı çalışır

best_w = study_blend.best_params
total_w = best_w['w_xgb'] + best_w['w_lgb'] + best_w['w_cat']
w_xgb = best_w['w_xgb'] / total_w
w_lgb = best_w['w_lgb'] / total_w
w_cat = best_w['w_cat'] / total_w

print(f"\n✅ İŞLEM TAMAM! Kusursuz Ağırlıklar Bulundu -> XGB: %{w_xgb*100:.1f}, LGB: %{w_lgb*100:.1f}, CAT: %{w_cat*100:.1f}")

# FİNAL TAHMİNLERİ
final_preds = (test_xgb * w_xgb) + (test_lgb * w_lgb) + (test_cat * w_cat)

sub = pd.DataFrame({'customerid': test['customerid'], 'churn': final_preds})
sub.to_csv('submission_v8_god_mode.csv', index=False)
print("🔥 'submission_v8_god_mode.csv' kaydedildi! Kaggle'a gönderebilirsiniz.")

[I 2026-05-20 11:22:52,854] A new study created in memory with name: no-name-ccab03f4-c07d-4de0-8040-3273d91e06e6


🎯 Adım 1/4: LightGBM optimize ediliyor...


[I 2026-05-20 11:22:54,841] Trial 0 finished with value: 0.8413775692304801 and parameters: {'n_estimators': 369, 'learning_rate': 0.03931507296313744, 'max_depth': 5, 'num_leaves': 20, 'reg_lambda': 1.693775254797269, 'subsample': 0.6981726661040557, 'colsample_bytree': 0.9446607226530722}. Best is trial 0 with value: 0.8413775692304801.
[I 2026-05-20 11:22:56,023] Trial 1 finished with value: 0.849098404038596 and parameters: {'n_estimators': 174, 'learning_rate': 0.03263538369432443, 'max_depth': 4, 'num_leaves': 29, 'reg_lambda': 6.595149784091036, 'subsample': 0.8357007309563464, 'colsample_bytree': 0.9810423805868872}. Best is trial 1 with value: 0.849098404038596.
[I 2026-05-20 11:22:57,364] Trial 2 finished with value: 0.8474653686665352 and parameters: {'n_estimators': 181, 'learning_rate': 0.013250921227345776, 'max_depth': 5, 'num_leaves': 25, 'reg_lambda': 4.9629703716208855, 'subsample': 0.8436625633063577, 'colsample_bytree': 0.7201160573844895}. Best is trial 1 with valu


🎯 Adım 2/4: CatBoost optimize ediliyor...


[I 2026-05-20 11:23:15,161] Trial 0 finished with value: 0.8486779242078877 and parameters: {'iterations': 247, 'learning_rate': 0.04477498291551908, 'depth': 4, 'l2_leaf_reg': 3.2197721113444535}. Best is trial 0 with value: 0.8486779242078877.
[I 2026-05-20 11:23:19,036] Trial 1 finished with value: 0.8495119947127225 and parameters: {'iterations': 364, 'learning_rate': 0.03019538362524386, 'depth': 4, 'l2_leaf_reg': 10.778044736835177}. Best is trial 1 with value: 0.8495119947127225.
[I 2026-05-20 11:23:20,726] Trial 2 finished with value: 0.8490379162181269 and parameters: {'iterations': 129, 'learning_rate': 0.04256379702606477, 'depth': 4, 'l2_leaf_reg': 7.039682328652972}. Best is trial 1 with value: 0.8495119947127225.
[I 2026-05-20 11:23:26,217] Trial 3 finished with value: 0.8477895893234318 and parameters: {'iterations': 382, 'learning_rate': 0.03104446954060426, 'depth': 5, 'l2_leaf_reg': 12.27219975453642}. Best is trial 1 with value: 0.8495119947127225.
[I 2026-05-20 11:2


🎯 Adım 3/4: Bulunan kusursuz ayarlarla modeller eğitiliyor...


[I 2026-05-20 11:24:09,883] A new study created in memory with name: no-name-abc77df3-f6bb-4c39-823e-aebc74e72c3d
[I 2026-05-20 11:24:09,890] Trial 0 finished with value: 0.8496916111609853 and parameters: {'w_xgb': 0.6830866663530499, 'w_lgb': 0.25235540834424486, 'w_cat': 0.5832188430310444}. Best is trial 0 with value: 0.8496916111609853.
[I 2026-05-20 11:24:09,895] Trial 1 finished with value: 0.8492788638297425 and parameters: {'w_xgb': 0.8651752794210801, 'w_lgb': 0.3439653084299902, 'w_cat': 0.08278971969931537}. Best is trial 0 with value: 0.8496916111609853.
[I 2026-05-20 11:24:09,900] Trial 2 finished with value: 0.849484914278973 and parameters: {'w_xgb': 0.7954907163622655, 'w_lgb': 0.8804555964986965, 'w_cat': 0.3697099133510209}. Best is trial 0 with value: 0.8496916111609853.
[I 2026-05-20 11:24:09,906] Trial 3 finished with value: 0.8496062820337744 and parameters: {'w_xgb': 0.192861754398449, 'w_lgb': 0.753586631951337, 'w_cat': 0.4024846068110495}. Best is trial 0 wit


🎯 Adım 4/4: Optuna en yüksek skoru verecek birleştirme oranlarını (Ağırlıkları) hesaplıyor...


[I 2026-05-20 11:24:10,097] Trial 21 finished with value: 0.8497617491178212 and parameters: {'w_xgb': 0.06745668910795724, 'w_lgb': 0.7749185066998439, 'w_cat': 0.8567502293295174}. Best is trial 21 with value: 0.8497617491178212.
[I 2026-05-20 11:24:10,108] Trial 22 finished with value: 0.8497842126569922 and parameters: {'w_xgb': 0.16837446194008487, 'w_lgb': 0.5679484254050485, 'w_cat': 0.8805077205134659}. Best is trial 22 with value: 0.8497842126569922.
[I 2026-05-20 11:24:10,120] Trial 23 finished with value: 0.8497800108439099 and parameters: {'w_xgb': 0.16892723810406246, 'w_lgb': 0.46452269199521806, 'w_cat': 0.873513764899308}. Best is trial 22 with value: 0.8497842126569922.
[I 2026-05-20 11:24:10,132] Trial 24 finished with value: 0.8497771018963913 and parameters: {'w_xgb': 0.13195018996075666, 'w_lgb': 0.42376900882034924, 'w_cat': 0.8908014512443941}. Best is trial 22 with value: 0.8497842126569922.
[I 2026-05-20 11:24:10,145] Trial 25 finished with value: 0.84977564742


✅ İŞLEM TAMAM! Kusursuz Ağırlıklar Bulundu -> XGB: %5.9, LGB: %37.6, CAT: %56.4
🔥 'submission_v8_god_mode.csv' kaydedildi! Kaggle'a gönderebilirsiniz.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingClassifier

# =========================
# 1) Dosyaları oku
# =========================

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

sub_v5 = pd.read_csv("submission_v5_optuna.csv")
sub_v6 = pd.read_csv("submission_v6_final.csv")
sub_v7 = pd.read_csv("submission_v7.csv")
sub_v8 = pd.read_csv("submission_v8_god_mode.csv")

print(train.shape, test.shape)
print(sub_v5.head())

# =========================
# 2) ID ve target kolonlarını bul
# =========================

possible_targets = ["target", "Target", "label", "Label", "churn", "Churn", "Exited", "is_churn"]

target_col = None
for col in possible_targets:
    if col in train.columns:
        target_col = col
        break

if target_col is None:
    # train'de olup test'te olmayan kolon genelde target olur
    diff_cols = [c for c in train.columns if c not in test.columns]
    if len(diff_cols) == 1:
        target_col = diff_cols[0]
    else:
        raise ValueError(f"Target kolonunu bulamadım. Train-test farkları: {diff_cols}")

possible_ids = ["id", "ID", "Id", "customer_id", "CustomerId", "PassengerId"]

id_col = None
for col in possible_ids:
    if col in test.columns:
        id_col = col
        break

if id_col is None:
    # submission'daki ilk kolon genelde id
    id_col = sub_v5.columns[0]

pred_col = sub_v5.columns[-1]

print("ID kolon:", id_col)
print("Target kolon:", target_col)
print("Prediction kolon:", pred_col)

# =========================
# 3) Feature hazırlığı
# =========================

y = train[target_col].copy()

drop_cols = [target_col]
if id_col in train.columns:
    drop_cols.append(id_col)

X = train.drop(columns=drop_cols, errors="ignore")
X_test = test.drop(columns=[id_col], errors="ignore")

# Train-test kolonlarını eşitle
common_cols = [c for c in X.columns if c in X_test.columns]
X = X[common_cols]
X_test = X_test[common_cols]

cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

print("Kategorik kolonlar:", cat_cols)
print("Sayısal kolon sayısı:", len(num_cols))

# Kategorikleri encode et
if len(cat_cols) > 0:
    enc = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1,
        encoded_missing_value=-1
    )

    X_cat = enc.fit_transform(X[cat_cols].astype(str))
    X_test_cat = enc.transform(X_test[cat_cols].astype(str))

    X_cat = pd.DataFrame(X_cat, columns=cat_cols, index=X.index)
    X_test_cat = pd.DataFrame(X_test_cat, columns=cat_cols, index=X_test.index)

    X_num = X[num_cols].reset_index(drop=True)
    X_test_num = X_test[num_cols].reset_index(drop=True)

    X_model = pd.concat([X_num, X_cat.reset_index(drop=True)], axis=1)
    X_test_model = pd.concat([X_test_num, X_test_cat.reset_index(drop=True)], axis=1)
else:
    X_model = X.copy()
    X_test_model = X_test.copy()

X_model = X_model.fillna(-999)
X_test_model = X_test_model.fillna(-999)

# =========================
# 4) CV modeli üret
# =========================

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof = np.zeros(len(train))
test_preds = np.zeros(len(test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_model, y), 1):
    X_tr, X_val = X_model.iloc[tr_idx], X_model.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model = HistGradientBoostingClassifier(
        learning_rate=0.035,
        max_iter=450,
        max_leaf_nodes=31,
        l2_regularization=0.05,
        random_state=42 + fold
    )

    model.fit(X_tr, y_tr)

    val_pred = model.predict_proba(X_val)[:, 1]
    test_pred = model.predict_proba(X_test_model)[:, 1]

    oof[val_idx] = val_pred
    test_preds += test_pred / skf.n_splits

    fold_auc = roc_auc_score(y_val, val_pred)
    print(f"Fold {fold} AUC:", round(fold_auc, 5))

cv_auc = roc_auc_score(y, oof)
print("OOF CV AUC:", round(cv_auc, 5))

# =========================
# 5) Blend dosyalarını üret
# =========================

v5 = sub_v5[pred_col].values
v6 = sub_v6[pred_col].values
v7 = sub_v7[pred_col].values
v8 = sub_v8[pred_col].values
cv_pred = test_preds

def make_submission(preds, filename):
    sub = sub_v5.copy()
    sub[pred_col] = np.clip(preds, 0, 1)
    sub.to_csv(filename, index=False)
    print(filename, "kaydedildi.")
    print(sub[pred_col].describe())
    return sub

# 1) En önerdiğim: v5 ağırlıklı, küçük CV dokunuşu
v9_safe = 0.90 * v5 + 0.10 * cv_pred
make_submission(v9_safe, "submission_v9_safe_90v5_10cv.csv")

# 2) Biraz daha agresif
v9_balanced = 0.80 * v5 + 0.20 * cv_pred
make_submission(v9_balanced, "submission_v9_balanced_80v5_20cv.csv")

# 3) Ultra güvenli: eski güçlü submission blend
v9_ultra_safe = 0.70 * v5 + 0.20 * v7 + 0.10 * v6
make_submission(v9_ultra_safe, "submission_v9_ultra_safe_v5v7v6.csv")

# 4) Rank tabanlı güvenli blend
def rank_normalize(x):
    return pd.Series(x).rank(method="average").values / len(x)

v5_rank = rank_normalize(v5)
cv_rank = rank_normalize(cv_pred)

rank_blend = 0.92 * v5_rank + 0.08 * cv_rank

# v5 dağılımına geri map et
order = np.argsort(rank_blend)
v5_sorted = np.sort(v5)

rank_qmap = np.zeros_like(v5)
rank_qmap[order] = v5_sorted

make_submission(rank_qmap, "submission_v9_rank_qmap_safe.csv")

# =========================
# 6) v5'e göre fark kontrolü
# =========================

candidates = {
    "v9_safe_90v5_10cv": v9_safe,
    "v9_balanced_80v5_20cv": v9_balanced,
    "v9_ultra_safe_v5v7v6": v9_ultra_safe,
    "v9_rank_qmap_safe": rank_qmap,
    "v8": v8,
    "v7": v7
}

print("\n--- v5'e göre fark analizi ---")

for name, preds in candidates.items():
    corr = np.corrcoef(v5, preds)[0, 1]
    mean_abs_diff = np.mean(np.abs(v5 - preds))
    class_changes = np.sum((v5 >= 0.5) != (preds >= 0.5))

    print(
        name,
        "| corr:", round(corr, 6),
        "| mean_abs_diff:", round(mean_abs_diff, 6),
        "| class_changes:", class_changes
    )

(5634, 21) (1409, 20)
   customerid     churn
0  4376-KFVRS  0.047577
1  2754-SDJRD  0.695106
2  9917-KWRBE  0.085691
3  0365-GXEZS  0.308817
4  9385-NXKDA  0.038333
ID kolon: customerid
Target kolon: churn
Prediction kolon: churn
Kategorik kolonlar: ['gender', 'partner', 'dependents', 'phoneservice', 'multiplelines', 'internetservice', 'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport', 'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling', 'paymentmethod']
Sayısal kolon sayısı: 4
Fold 1 AUC: 0.8317
Fold 2 AUC: 0.82003
Fold 3 AUC: 0.83759
Fold 4 AUC: 0.84317
Fold 5 AUC: 0.83315
OOF CV AUC: 0.83286
submission_v9_safe_90v5_10cv.csv kaydedildi.
count    1409.000000
mean        0.265248
std         0.234373
min         0.013171
25%         0.053718
50%         0.192532
75%         0.438414
max         0.875411
Name: churn, dtype: float64
submission_v9_balanced_80v5_20cv.csv kaydedildi.
count    1409.000000
mean        0.264332
std         0.237897
min         0